# App Rating Prediction

**Tabular Regression Project** — Predict mobile app store rating from app metadata and engagement features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `app_rating`

## 2 · Project Overview

We predict **mobile app store ratings** (1-5 scale) from app metadata and engagement metrics. App rating prediction helps developers understand what drives user satisfaction and guides app store optimization (ASO) strategies.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given an app's size, review count, installs, price, category, content rating, update recency, in-app purchases, screenshots, and description length, predict its **average user rating** (1-5).

## 5 · Why This Project Matters

- **App Store Optimization (ASO)** relies on understanding rating drivers.
- Higher ratings drive more downloads — a positive feedback loop.
- Developers can prioritize improvements that most impact ratings.
- **Update frequency** and **listing quality** (screenshots, descriptions) are actionable.
- A common data science interview problem with clear business relevance.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,500 |
| **Features** | 10 (size, reviews, installs, price, category, content rating, days since update, IAP, screenshots, description length) |
| **Target** | `app_rating` (continuous, 1.0-5.0) |
| **Categorical** | category (8), content_rating (3) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "app_rating"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8500
app_size_mb = np.random.lognormal(3, 1, n).clip(1, 500)
num_reviews = np.random.lognormal(6, 2, n).clip(1, 5000000).astype(int)
installs = np.random.lognormal(10, 2, n).clip(100, 1e9).astype(int)
price = np.random.choice([0.0] * 7 + [0.99, 1.99, 2.99, 4.99, 9.99], n)
category = np.random.choice(["games", "social", "productivity", "education",
                              "health", "entertainment", "finance", "tools"], n)
content_rating = np.random.choice(["everyone", "teen", "mature"], n, p=[0.6, 0.25, 0.15])
days_since_update = np.random.randint(1, 1000, n)
has_iap = np.random.choice([0, 1], n, p=[0.4, 0.6])
num_screenshots = np.random.randint(2, 10, n)
description_length = np.random.randint(50, 5000, n)

rating = (3.5 + np.log1p(num_reviews) * 0.05
          - days_since_update * 0.0005
          + num_screenshots * 0.03
          + np.log1p(description_length) * 0.05
          - app_size_mb * 0.001
          + (price > 0).astype(float) * 0.1
          + np.random.normal(0, 0.3, n))
rating = rating.clip(1.0, 5.0)

df = pd.DataFrame({
    "app_size_mb": app_size_mb, "num_reviews": num_reviews,
    "installs": installs, "price_usd": price,
    "category": category, "content_rating": content_rating,
    "days_since_update": days_since_update, "has_in_app_purchases": has_iap,
    "num_screenshots": num_screenshots, "description_length": description_length,
    "app_rating": rating
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["num_reviews", "installs", "days_since_update",
                          "app_size_mb", "num_screenshots", "description_length"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["app_rating"], alpha=0.15, s=6)
    ax.set_xlabel(col); ax.set_ylabel("app_rating"); ax.set_title(f"Rating vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("category")["app_rating"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Mean Rating by Category")
df.groupby("content_rating")["app_rating"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Mean Rating by Content Rating")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `app_rating`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("Rating (1-5)")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():.3f}, Median: {df[TARGET].median():.3f}")
print(f"Std: {df[TARGET].std():.3f}, Range: {df[TARGET].min():.2f} - {df[TARGET].max():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["log_reviews"] = np.log1p(X_train["num_reviews"])
X_test["log_reviews"] = np.log1p(X_test["num_reviews"])

X_train["log_installs"] = np.log1p(X_train["installs"])
X_test["log_installs"] = np.log1p(X_test["installs"])

X_train["reviews_per_install"] = X_train["num_reviews"] / (X_train["installs"] + 1)
X_test["reviews_per_install"] = X_test["num_reviews"] / (X_test["installs"] + 1)

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Review count** (log-transformed) positively correlates with rating — well-reviewed apps tend to maintain quality.
- **Days since update** has a negative effect — stale apps lose user trust.
- **Number of screenshots** and **description length** reflect developer effort and correlate with higher ratings.
- **App size** has a small negative effect — bloated apps frustrate users.
- **Paid apps** (price > 0) have slightly higher ratings — selection bias (paying users expect and tolerate more).

**Business takeaway:** Regularly update your app, invest in listing quality (screenshots, descriptions), and keep app size lean. These are the most actionable levers for rating improvement.

## 26 · Limitations

1. Synthetic data — real app ratings have complex dynamics (review bombing, bot reviews).
2. No text analysis of actual reviews — sentiment analysis would add signal.
3. Ratings are bounded [1, 5] — regression near boundaries may misbehave.
4. No temporal trend — app quality expectations change over time.
5. Survivorship bias — only apps still listed are observed.

## 27 · How to Improve This Project

1. Use real Google Play or App Store data (scraped or via API).
2. Add NLP features from review text (sentiment, topic distribution).
3. Include competitor rating comparison within category.
4. Add crash rate and ANR (Application Not Responding) metrics.
5. Model as ordinal regression to respect the 1-5 scale.

## 28 · Production Considerations

- Monitor rating prediction vs. actual weekly.
- Alert developers when predicted rating drops below threshold.
- Use as a feature in app recommendation engines.
- Retrain quarterly as user expectations evolve.

## 29 · Common Mistakes

1. Not log-transforming review count and installs (extreme right skew).
2. Treating ratings as unbounded — predictions outside [1, 5] are meaningless.
3. Ignoring the selection bias of paid vs. free apps.
4. Not accounting for category-specific rating norms (games tend lower).
5. Using installs as a predictor without controlling for age (older apps have more installs).

## 30 · Mini Challenge / Exercises

1. Clip predictions to [1, 5] — does it improve RMSE?
2. Remove `num_reviews` and `installs` — how much predictive power remains?
3. Add a `is_free` binary feature — does it outperform raw price?
4. Train only on games — does the model generalize to productivity apps?
5. Try ordinal classification (rounding to nearest 0.5) instead of regression.

## 31 · Final Summary / Key Takeaways

1. **Review count** and **update recency** are the top rating predictors.
2. **Listing quality** (screenshots, descriptions) reflects developer investment.
3. **Log transforms** of count features are essential for model performance.
4. **Gradient boosting** captures non-linear effects well on this bounded target.
5. **App size** has a small but consistent negative effect.
6. **Real deployment** needs review text analysis and temporal feature tracking.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))